# Reinforcement Learning Fine-tuning with DPO using Unsloth

This notebook demonstrates Direct Preference Optimization (DPO) fine-tuning using Unsloth. DPO is a reinforcement learning technique that trains models based on human preferences.

## What we'll cover:
- Installing Unsloth and dependencies
- Loading a model for DPO training
- Using a dataset with preferred and rejected responses
- Training with DPO to align the model with human preferences
- Testing the aligned model

## About DPO (Direct Preference Optimization):
DPO is a reinforcement learning method that:
- Trains models using paired examples: preferred vs rejected responses
- Directly optimizes the model to produce preferred outputs
- Simpler than traditional RLHF (doesn't require a separate reward model)
- Effective for alignment and safety

## Dataset Format:
DPO requires a dataset with three columns:
- `prompt`: The input/instruction
- `chosen`: The preferred/better response
- `rejected`: The less preferred/worse response

The model learns to increase the probability of chosen responses and decrease the probability of rejected responses.

In [1]:
# Install Unsloth and dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-jnlr3x2p/unsloth_24813f061379411698ab60344d9c1bb3
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-jnlr3x2p/unsloth_24813f061379411698ab60344d9c1bb3
  Resolved https://github.com/unslothai/unsloth.git to commit 341ce85864d191e4a6b7c447b9167c1faf5e20d3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.5/283.5 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 112.1 MB/s eta 0:00:00

## Import Required Libraries

Import the necessary libraries for DPO training. We'll use:
- `FastLanguageModel` from Unsloth for model loading
- `DPOTrainer` from TRL for Direct Preference Optimization
- `DPOConfig` for training configuration
- Standard PyTorch and datasets libraries

In [2]:
# Import necessary libraries
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch
from trl import DPOTrainer, DPOConfig
from datasets import load_dataset

# Check if GPU is available
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU Available: True
GPU Name: NVIDIA L4
GPU Memory: 23.80 GB


## Model Configuration for DPO Training

For DPO (Direct Preference Optimization), we'll use LoRA with 4-bit quantization for memory efficiency.

Configuration parameters:
- `max_seq_length`: Maximum sequence length (2048 tokens)
- `dtype`: Data type (None for auto-detection)
- `load_in_4bit`: True for memory-efficient 4-bit quantization
- `model_name`: We'll use SmolLM2 135M for consistency with previous colabs

DPO works best with LoRA adapters since it's a preference-based fine-tuning method that typically requires smaller parameter updates compared to full fine-tuning.

In [3]:
# Configuration parameters for DPO training
max_seq_length = 2048  # Maximum sequence length
dtype = None  # Auto-detect dtype
load_in_4bit = True  # Use 4-bit quantization for efficiency

# Model selection - using SmolLM2 135M
model_name = "unsloth/SmolLM2-135M-Instruct"

print(f"Model: {model_name}")
print(f"Max Sequence Length: {max_seq_length}")
print(f"4-bit Quantization: {load_in_4bit}")
print("DPO training with LoRA adapters")

Model: unsloth/SmolLM2-135M-Instruct
Max Sequence Length: 2048
4-bit Quantization: True
DPO training with LoRA adapters


## Load Model and Configure LoRA Adapters for DPO

We load the model with 4-bit quantization and add LoRA adapters, similar to Colab 2.

For DPO training, we need:
- A base model with LoRA adapters
- The model will learn to prefer chosen responses over rejected ones
- LoRA makes this efficient by only updating adapter parameters

The configuration is similar to standard LoRA fine-tuning, but the training objective (DPO loss) is different.

In [4]:
# Load model and tokenizer with 4-bit quantization
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

print("Model loaded successfully with LoRA adapters for DPO!")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
trainable_percentage = 100 * sum(p.numel() for p in model.parameters() if p.requires_grad) / sum(p.numel() for p in model.parameters())
print(f"Trainable percentage: {trainable_percentage:.2f}%")

==((====))==  Unsloth 2025.11.3: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/158 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/29.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/423 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Unsloth 2025.11.3 patched 30 layers with 30 QKV layers, 30 O layers and 30 MLP layers.


Model loaded successfully with LoRA adapters for DPO!
Total parameters: 86,315,904
Trainable parameters: 4,884,480
Trainable percentage: 5.66%


## Load DPO Training Dataset

For DPO training, we need a dataset with preference pairs: chosen (preferred) and rejected (not preferred) responses.

We'll use a popular DPO dataset that contains:
- `prompt`: The instruction or query
- `chosen`: The preferred/better response
- `rejected`: The less preferred/worse response

Common DPO datasets include:
- `Anthropic/hh-rlhf`: Human preference data
- `Intel/orca_dpo_pairs`: Instruction following preferences
- `argilla/ultrafeedback-binarized-preferences-cleaned`: High-quality preference data

We'll use a subset for quick training demonstration.

In [7]:
# Load a DPO dataset with preference pairs
# Using Intel's orca_dpo_pairs dataset (small and high-quality)
dataset = load_dataset("Intel/orca_dpo_pairs", split="train")

# Take a smaller subset for quick training (first 1000 examples)
dataset = dataset.select(range(min(1000, len(dataset))))

print(f"Dataset loaded successfully!")
print(f"Number of examples: {len(dataset)}")
print(f"Dataset columns: {dataset.column_names}")
print("\nFirst example:")
print(f"System: {dataset[0]['system'][:200]}...")
print(f"\nQuestion: {dataset[0]['question'][:200]}...")
print(f"\nChosen response: {dataset[0]['chosen'][:200]}...")
print(f"\nRejected response: {dataset[0]['rejected'][:200]}...")

Dataset loaded successfully!
Number of examples: 1000
Dataset columns: ['system', 'question', 'chosen', 'rejected']

First example:
System: ...

Question: You will be given a definition of a task first, then some input of the task.
This task is about using the specified sentence and converting the sentence to Resource Description Framework (RDF) triplet...

Chosen response: [
  ["AFC Ajax (amateurs)", "has ground", "Sportpark De Toekomst"],
  ["Ajax Youth Academy", "plays at", "Sportpark De Toekomst"]
]...

Rejected response:  Sure, I'd be happy to help! Here are the RDF triplets for the input sentence:

[AFC Ajax (amateurs), hasGround, Sportpark De Toekomst]
[Ajax Youth Academy, playsAt, Sportpark De Toekomst]

Explanatio...


## Format Dataset for DPO Training

We need to format the dataset to match the expected DPO format. The DPOTrainer expects:
- `prompt`: The input/instruction
- `chosen`: The preferred response
- `rejected`: The less preferred response

Since our dataset has `system`, `question`, `chosen`, and `rejected`, we'll combine system and question to create the prompt.

In [10]:
# Format dataset for DPO training
def format_dpo_dataset(examples):
    """
    Format the dataset for DPO training.
    Combine system and question into a prompt.
    """
    prompts = []
    for system, question in zip(examples['system'], examples['question']):
        # Combine system and question into a single prompt
        prompt = f"{system}\n\n{question}"
        prompts.append(prompt)

    return {
        'prompt': prompts,
        'chosen': examples['chosen'],
        'rejected': examples['rejected']
    }

# Apply formatting
formatted_dataset = dataset.map(
    format_dpo_dataset,
    batched=True,
    remove_columns=['system', 'question']
)

print("Dataset formatted for DPO training!")
print(f"Columns after formatting: {formatted_dataset.column_names}")
print("\nFormatted example:")
print(f"Prompt: {formatted_dataset[0]['prompt'][:300]}...")
print(f"\nChosen: {formatted_dataset[0]['chosen'][:200]}...")
print(f"\nRejected: {formatted_dataset[0]['rejected'][:200]}...")

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset formatted for DPO training!
Columns after formatting: ['chosen', 'rejected', 'prompt']

Formatted example:
Prompt: 

You will be given a definition of a task first, then some input of the task.
This task is about using the specified sentence and converting the sentence to Resource Description Framework (RDF) triplets of the form (subject, predicate object). The RDF triplets generated must be such that the triple...

Chosen: [
  ["AFC Ajax (amateurs)", "has ground", "Sportpark De Toekomst"],
  ["Ajax Youth Academy", "plays at", "Sportpark De Toekomst"]
]...

Rejected:  Sure, I'd be happy to help! Here are the RDF triplets for the input sentence:

[AFC Ajax (amateurs), hasGround, Sportpark De Toekomst]
[Ajax Youth Academy, playsAt, Sportpark De Toekomst]

Explanatio...


## Setup DPO Training Configuration

DPO (Direct Preference Optimization) uses `DPOConfig` instead of standard `TrainingArguments`.

Key DPO-specific parameters:
- `beta`: The temperature parameter for DPO loss (default 0.1). Higher values make the model less sensitive to preference differences
- `loss_type`: Type of DPO loss ("sigmoid" is standard)
- `max_prompt_length`: Maximum length for prompts
- `max_length`: Maximum total length (prompt + response)

Standard training parameters:
- `per_device_train_batch_size`: Batch size per device
- `gradient_accumulation_steps`: Gradient accumulation
- `learning_rate`: Learning rate for optimizer
- `max_steps`: Total training steps
- `fp16/bf16`: Mixed precision training

In [11]:
# Setup DPO training configuration
dpo_config = DPOConfig(
    # DPO-specific parameters
    beta=0.1,  # Temperature for DPO loss
    loss_type="sigmoid",  # DPO loss type
    max_prompt_length=512,  # Max prompt length
    max_length=1024,  # Max total length (prompt + response)

    # Standard training parameters
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=60,  # Reduced for quick demo
    learning_rate=5e-5,  # Lower learning rate for DPO
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=1,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    output_dir="outputs_dpo",
    report_to="none",
)

print("DPO training configuration set!")
print(f"Beta (temperature): {dpo_config.beta}")
print(f"Loss type: {dpo_config.loss_type}")
print(f"Max steps: {dpo_config.max_steps}")
print(f"Learning rate: {dpo_config.learning_rate}")

DPO training configuration set!
Beta (temperature): 0.1
Loss type: sigmoid
Max steps: 60
Learning rate: 5e-05


## Initialize the DPO Trainer

The `DPOTrainer` handles the preference optimization training process. It:
- Takes the model with LoRA adapters
- Uses the formatted dataset with prompt, chosen, and rejected columns
- Optimizes the model to prefer chosen responses over rejected ones
- Uses the DPO loss function to update model parameters

Key differences from SFTTrainer:
- Requires both chosen and rejected responses
- Uses a different loss function (DPO loss)
- Trains the model to maximize the likelihood difference between chosen and rejected responses

In [12]:
# Initialize the DPO Trainer
dpo_trainer = DPOTrainer(
    model=model,
    args=dpo_config,
    train_dataset=formatted_dataset,
    tokenizer=tokenizer,
)

print("DPO Trainer initialized successfully!")
print(f"Training dataset size: {len(dpo_trainer.train_dataset)}")
print(f"Model will learn to prefer 'chosen' responses over 'rejected' responses")

Extracting prompt in train dataset (num_proc=16):   0%|          | 0/1000 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=16):   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=16):   0%|          | 0/1000 [00:00<?, ? examples/s]

DPO Trainer initialized successfully!
Training dataset size: 1000
Model will learn to prefer 'chosen' responses over 'rejected' responses


## Train the Model with DPO

Now we start the DPO training process. The trainer will:
- Process pairs of chosen and rejected responses
- Compute the DPO loss (preference loss)
- Update model parameters to increase probability of chosen responses
- Decrease probability of rejected responses
- Log training metrics including DPO-specific losses

Expected behavior:
- The model learns human preferences through contrastive learning
- Training optimizes the policy to align with preferred behaviors
- The loss should decrease as the model learns to distinguish preferences

This is different from supervised fine-tuning - instead of just learning to generate text, the model learns which responses are better.

In [13]:
# Start DPO training
print("Starting DPO training...")
trainer_stats = dpo_trainer.train()

print("\nDPO Training completed!")
print(f"Training loss: {trainer_stats.training_loss:.4f}")
print(f"Training time: {trainer_stats.metrics['train_runtime']:.2f} seconds")
print(f"Training samples per second: {trainer_stats.metrics['train_samples_per_second']:.2f}")
print("\nThe model has been aligned with human preferences!")

The model is already on multiple devices. Skipping the move to device specified in `args`.


Starting DPO training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 4,884,480 of 139,400,064 (3.50% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected,eval_logits / chosen,eval_logits / rejected,nll_loss
1,0.693100,0.000000,0.000000,0.000000,0.000000,-246.099350,-288.093079,6.070113,4.730857,0,0,0
2,0.693100,0.000000,0.000000,0.000000,0.000000,-307.540100,-372.139465,4.404870,3.481539,No Log,No Log,No Log
3,0.692600,0.029289,0.025522,0.500000,0.003768,-394.766846,-389.724304,4.190316,3.130998,No Log,No Log,No Log
4,0.694400,-0.009520,-0.009584,0.500000,0.000064,-174.440765,-362.736084,3.545520,3.775225,No Log,No Log,No Log
5,0.693700,0.026637,0.025501,0.500000,0.001136,-336.458405,-408.196198,6.366066,5.395390,No Log,No Log,No Log
6,0.693900,-0.024646,-0.026315,0.625000,0.001669,-379.216919,-391.498505,4.034572,3.986002,No Log,No Log,No Log
7,0.707100,-0.030276,-0.003207,0.125000,-0.027069,-213.501419,-335.349915,2.247008,3.002683,No Log,No Log,No Log
8,0.702600,-0.013698,0.003833,0.625000,-0.017532,-256.149017,-340.433350,2.727810,2.492497,No Log,No Log,No Log
9,0.684400,-0.015984,-0.033820,0.500000,0.017836,-110.891441,-318.576202,3.585222,3.178457,No Log,No Log,No Log
10,0.703700,-0.046836,-0.026964,0.500000,-0.019872,-450.105835,-317.133789,5.548703,4.512669,No Log,No Log,No Log



DPO Training completed!
Training loss: 0.5866
Training time: 135.21 seconds
Training samples per second: 3.55

The model has been aligned with human preferences!


## Test the DPO Fine-tuned Model

After DPO training, the model should generate responses that are more aligned with human preferences.

We'll test the model with a sample prompt to see how it responds. The model should:
- Generate higher quality responses
- Avoid patterns similar to rejected responses
- Follow instructions better
- Be more aligned with helpful, harmless, and honest behavior

Let's compare the output with the base model behavior.

In [14]:
# Enable fast inference mode
FastLanguageModel.for_inference(model)

# Create a test prompt
test_prompt = """You are a helpful assistant.

Question: What are three tips for staying healthy?

Answer:"""

# Tokenize the input
inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")

# Generate response
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

print("DPO-aligned model response:")
print("-" * 50)
outputs = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=150,
    use_cache=True,
    temperature=0.7,
    top_p=0.9
)
print("-" * 50)
print("\nThe model should now generate responses more aligned with preferred behaviors!")

DPO-aligned model response:
--------------------------------------------------
 To stay healthy, you should eat a balanced diet, exercise regularly, and get enough sleep.<|im_end|>
--------------------------------------------------

The model should now generate responses more aligned with preferred behaviors!
